<center><h1 style="color:#D4AF37"> ⚡⚡ AEMR Data Analysis ⚡⚡</h1>

<img src = "https://images.squarespace-cdn.com/content/v1/551972d8e4b0d571edc2e2c8/8f3abd33-0916-4415-9b24-593eea1d1e3c/Screen+Shot+2022-04-20+at+12.41.24+PM.png">

<h1 style="color:#D4AF37"> What's the Business Problem? </h1>

The American Energy Market Regulator (AEMR) is responsible for looking after the
United States of America’s domestic energy network. The regulator’s responsibility is to
ensure that America’s energy network remains reliable with minimal disruptions, which
are known as outages. 

There are four key types of outages:

● Consequential

● Forced 

● Opportunistic 

● Planned 

Recently, the AEMR management team has been increasingly aware of a large number of energy providers that submitted outages over the 2016 and 2017 calendar years. The management team has expressed a desire to have the following two areas of concern addressed:

<b> A) Energy Stability and Market Outages
    <p>
B) Energy Losses and Market Reliability </b>

As an analyst, I will address the two immediate areas of concern identified by management. Using SQL, I will extract and analyze data from the database to answer the key business questions. In addition to these priorities, I will explore further insights and trends that may be valuable and of interest to the management team.

<h3 style="color:#D4AF37"> SQL Database Loading and Setup for AEMR Analysis </h3>

In [659]:
!pip install ipython-sql
!pip install prettytable==3.12

In [660]:
import requests
from IPython.core.magic import register_line_magic
from IPython.display import HTML
import sqlite3

@register_line_magic
def load_sqlite_db(url):
    response = requests.get(url)

    if response.status_code == 200:
        with open('temp_db_file.db', 'wb') as file:
            file.write(response.content)
        #print('SQLite database file downloaded successfully.')
    else:
        print('Failed to download the SQLite database file.')

sqlite_db_url = 'https://raw.githubusercontent.com/chrishuisb1990/practice_datasets/main/AEMR.db'

%load_sqlite_db $sqlite_db_url

%sql sqlite:///temp_db_file.db

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [661]:
%%sql
SELECT
*
FROM AEMR_Outage_Table
LIMIT 10;

 * sqlite:///temp_db_file.db
Done.


EventID,Start_Time,End_Time,Year,Month,Facility_Code,Participant_Code,Status,Outage_Reason,Energy_Lost_MW,Description_Of_Outage
1,2017-12-28 06:00,2017-12-28 10:00,2017,12,DNHR_DENMARK_WF1,DNHR,Approved,Consequential,1.44,a network outage in Denmark caused the windfarm to trip at 6:26 am
2,2017-12-26 09:00,2017-12-27 00:00,2017,12,COLLGAR_WF1,COLLGAR,Approved,Forced,30.0,Forced Outage - BS on Overhead line pole S-81 - CG10/11
3,2017-12-31 09:00,2017-12-31 09:00,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,5.702999999999999,Under generation - ambient conditions
4,2017-12-31 08:30,2017-12-31 08:30,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,5.085,Under generation - ambient conditions
5,2017-12-31 08:00,2017-12-31 08:00,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,5.02,Under generation - ambient conditions
6,2017-12-30 23:30,2017-12-30 23:30,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,5.79,Under generation - ambient conditions
7,2017-12-30 23:00,2017-12-30 23:00,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,5.412000000000001,Under generation - ambient conditions
8,2017-12-30 22:30,2017-12-30 22:30,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,4.735,Under generation - ambient conditions
9,2017-12-29 23:30,2017-12-29 23:30,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,4.245,Under generation - ambient conditions
10,2017-12-29 23:00,2017-12-29 23:00,2017,12,AURICON_PNJ_U1,AURICON,Approved,Forced,3.9619999999999997,Under generation - ambient conditions


<h3 style="color:#D4AF37"> ⚡ Part I. Energy Stability & Market Outages ⚡ </h3>

Energy stability is one of the key themes the AEMR management team cares about. To ensure energy security and reliability, AEMR needs to understand the following:
<p>

    1. What are the most common outage types and how long do they tend to last?
    2. How frequently do the outages occur? 
    3. Are there any energy providers which have more outages than their peers which may be indicative of being unreliable? 
    
<p>


<h3 style="color:#D4AF37"> Question 1 </h3> 
<b style="color:#1708B8"> How many outages are approved in comparison to those that are canceled? </b>

In [667]:
%%sql
SELECT 
    Status,
    Year,
    COUNT(*) AS Total_Number_Outages
       
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
GROUP BY Year, Status
ORDER BY Year, Status

 * sqlite:///temp_db_file.db
Done.


Status,Year,Total_Number_Outages
Accepted,2016,3
Approved,2016,1931
Cancelled By Market Participant,2016,207
Cancelled By System Management,2016,1
Not Accepted,2016,3
Rejected,2016,1
Accepted,2017,6
Approved,2017,2171
Cancelled By Market Participant,2017,315
Cancelled By System Management,2017,5


<u style="color:Maroon">
Please note that for the remainder of this analysis, I will focus exclusively on outages where <b> Status = 'Approved'</b>. 
All cancelled or unapproved outages are excluded from consideration. Accordingly, every SQL query in this case study will consistently include the condition <b>WHERE Status = 'Approved'</b>.
</u>

<h3 style="color:#D4AF37"> Question 2 </h3> 
<b style="color:#1708B8"> How many approved outage events occurred across each outage type (Forced, Consequential, Scheduled, Opportunistic) during 2016 and 2017?</b>

In [671]:
%%sql
SELECT 
    Outage_Reason,
    Year,
    COUNT(*) AS Total_Number_Outages
       
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
    AND Status = 'Approved'
GROUP BY Year, Outage_Reason
ORDER BY Year, Outage_Reason
LIMIT 4

 * sqlite:///temp_db_file.db
Done.


Outage_Reason,Year,Total_Number_Outages
Consequential,2016,181
Forced,2016,1264
Opportunistic Maintenance (Planned),2016,106
Scheduled (Planned),2016,380


<h3 style="color:#D4AF37"> Question 3 </h3> 
<b style="color:#1708B8"> What monthly trends can be observed in outage events? Do certain months show higher outage activity or indicate greater reliability issues compared to others? </b>

In [674]:
%%sql
SELECT 
    Year,
    Month,
    COUNT(*) AS Total_Number_Outages
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Year, Month
ORDER BY Year, Month, Total_Number_Outages
LIMIT 5;


 * sqlite:///temp_db_file.db
Done.


Year,Month,Total_Number_Outages
2016,1,191
2016,2,227
2016,3,136
2016,4,134
2016,5,174


<h3 style="color:#D4AF37"> Question 4 </h3>
<b style="color:#1708B8"> For each participant and outage type during 2016–2017, what are the total number of approved outage events and the average outage duration (in days, rounded to two decimals)?
</b>

In [677]:
%%sql
SELECT 
    Participant_Code,
    Outage_Reason,
    Year,
    COUNT(*) AS Total_Number_Outage_Events,
    ROUND( AVG((ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time)))), 2) AS Average_Outage_Duration_In_Days
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Participant_Code, Outage_Reason, Year
ORDER BY 
    Total_Number_Outage_Events DESC,
    Outage_Reason,
    Year
LIMIT 5;

 * sqlite:///temp_db_file.db
Done.


Participant_Code,Outage_Reason,Year,Total_Number_Outage_Events,Average_Outage_Duration_In_Days
AURICON,Forced,2017,490,0.07
GW,Forced,2016,317,0.38
GW,Forced,2017,227,1.06
AURICON,Forced,2016,208,0.07
AUXC,Forced,2016,206,0.08


<h3 style="color:#D4AF37"> Question 5 </h3>
<b style="color:#1708B8"> How can participants be classified based on reliability metrics such as outage frequency and duration to assess their overall uptime performance?
</b>


<b>Risk classification is determined based on (1) Participant's average availability, (2) Participant's average availability and Total Number of outage events, and (3) Participant's average availability and Total Number of outage events excluding Forced outage .</b>

<b style="color:#cc0000"> Criteria 1 </b>

<b>
<ul>
  <li><span style="color:#800000;">High Risk</span> – On average, the participant is unavailable for more than 24 hours (1 day).</li>
  <li><span style="color:#cc6600;">Medium Risk</span> – On average, the participant is unavailable between 12 and 24 hours.</li>
  <li><span style="color:#006400;">Low Risk</span> – On average, the participant is unavailable for less than 12 hours.</li>
</ul>
</b>


In [681]:
%%sql
SELECT 
    Participant_Code,
    Outage_Reason,
    Year,
    COUNT(*) AS Total_Number_Outage_Events,
    ROUND(AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))), 2) AS Average_Outage_Duration_In_Days,
    CASE
        WHEN AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) > 1 THEN 'High Risk'
        WHEN AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) BETWEEN 0.5 AND 1 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Classification
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Participant_Code, Outage_Reason, Year
ORDER BY Total_Number_Outage_Events DESC, Outage_Reason, Year
LIMIT 3;


 * sqlite:///temp_db_file.db
Done.


Participant_Code,Outage_Reason,Year,Total_Number_Outage_Events,Average_Outage_Duration_In_Days,Risk_Classification
AURICON,Forced,2017,490,0.07,Low Risk
GW,Forced,2016,317,0.38,Low Risk
GW,Forced,2017,227,1.06,High Risk


<b style="color:#cc0000"> Criteria 2 </b>

<b>
<ul>
  <li><span style="color:#800000;">High Risk</span> – On average, the participant is unavailable for > 24 Hours (1 Day) OR the Total Number of Outage Events > 20.</li>
  <li><span style="color:#cc6600;">Medium Risk</span> – On average, the participant is unavailable between 12 and 24 Hours OR the Total Number of Outage Events is Between 10 and 20.</li>
  <li><span style="color:#006400;">Low Risk</span> – On average, the participant is unavailable for less than 12 Hours OR the Total Number of Outage Events.</li>
</ul>
</b>

In [684]:
%%sql
SELECT 
    Participant_Code,
    Outage_Reason,
    Year,
    COUNT(*) AS Total_Number_Outage_Events,
    ROUND(AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))), 2) AS Average_Outage_Duration_In_Days,
    CASE
        WHEN AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) > 1 THEN 'High Risk'
        WHEN AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) BETWEEN 0.5 AND 1 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Classification
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Participant_Code, Outage_Reason, Year
ORDER BY Total_Number_Outage_Events DESC, Outage_Reason, Year
LIMIT 3;


 * sqlite:///temp_db_file.db
Done.


Participant_Code,Outage_Reason,Year,Total_Number_Outage_Events,Average_Outage_Duration_In_Days,Risk_Classification
AURICON,Forced,2017,490,0.07,Low Risk
GW,Forced,2016,317,0.38,Low Risk
GW,Forced,2017,227,1.06,High Risk


In [686]:
%%sql
SELECT 
    Participant_Code,
    Outage_Reason,
    Year,
    COUNT(*) AS Total_Number_Outage_Events,
    ROUND(AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))), 2) AS Average_Outage_Duration_In_Days,
    CASE
        WHEN AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) > 1 
             OR COUNT(*) > 20 THEN 'High Risk'
        WHEN (AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) BETWEEN 0.5 AND 1) 
             OR (COUNT(*) BETWEEN 10 AND 20) THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Classification
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Participant_Code, Outage_Reason, Year
ORDER BY 
    Participant_Code, 
    Outage_Reason DESC, 
    Year	
LIMIT 10;


 * sqlite:///temp_db_file.db
Done.


Participant_Code,Outage_Reason,Year,Total_Number_Outage_Events,Average_Outage_Duration_In_Days,Risk_Classification
AURICON,Scheduled (Planned),2016,46,1.89,High Risk
AURICON,Scheduled (Planned),2017,45,1.45,High Risk
AURICON,Opportunistic Maintenance (Planned),2016,3,0.33,Low Risk
AURICON,Forced,2016,208,0.07,High Risk
AURICON,Forced,2017,490,0.07,High Risk
AURICON,Consequential,2016,41,0.13,High Risk
AURICON,Consequential,2017,42,0.21,High Risk
AUXC,Scheduled (Planned),2016,2,1.25,High Risk
AUXC,Scheduled (Planned),2017,1,2.88,High Risk
AUXC,Forced,2016,206,0.08,High Risk


<b style="color:#cc0000"> Criteria 3 </b>

<b>
<ul>
  <li><span style="color:#800000;">High Risk</span> – On average, the participant is unavailable for > 24 Hours (1 Day) OR the Total Number of Outage Events > 20.</li>
  <li><span style="color:#cc6600;">Medium Risk</span> – On average, the participant is unavailable between 12 and 24 Hours OR the Total Number of Outage Events is Between 10 and 20.</li>
  <li><span style="color:#006400;">Low Risk</span> – On average, the participant is unavailable for less than 12 Hours OR the Total Number of Outage Events.</li>
  <li> If Outage Type is not forced, then N/A
</ul>
</b>

In [689]:
%%sql
SELECT 
    Participant_Code,
    Outage_Reason,
    Year,
    COUNT(*) AS Total_Number_Outage_Events,
    ROUND(AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))), 2) AS Average_Outage_Duration_In_Days,
    CASE
        WHEN Outage_Reason <> 'Forced' THEN 'N/A'
        WHEN AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) > 1 
             OR COUNT(*) > 20 THEN 'High Risk'
        WHEN (AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))) BETWEEN 0.5 AND 1) 
             OR (COUNT(*) BETWEEN 10 AND 20) THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Classification
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Participant_Code, Outage_Reason, Year
ORDER BY 
    Participant_Code, 
    Outage_Reason DESC, 
    Year	
LIMIT 10;


 * sqlite:///temp_db_file.db
Done.


Participant_Code,Outage_Reason,Year,Total_Number_Outage_Events,Average_Outage_Duration_In_Days,Risk_Classification
AURICON,Scheduled (Planned),2016,46,1.89,N/A
AURICON,Scheduled (Planned),2017,45,1.45,N/A
AURICON,Opportunistic Maintenance (Planned),2016,3,0.33,N/A
AURICON,Forced,2016,208,0.07,High Risk
AURICON,Forced,2017,490,0.07,High Risk
AURICON,Consequential,2016,41,0.13,N/A
AURICON,Consequential,2017,42,0.21,N/A
AUXC,Scheduled (Planned),2016,2,1.25,N/A
AUXC,Scheduled (Planned),2017,1,2.88,N/A
AUXC,Forced,2016,206,0.08,High Risk


<h3 style="color:#D4AF37"> ⚡ Part II. Energy Losses & Market Reliability ⚡ </h3>

<b> Forced outages pose a significant reliability risk because providers fail to deliver the energy they committed to supplying. If several providers are forced offline at once, it can threaten overall grid stability—something the AEMR must closely monitor.</b>

In the next section, I will analyze forced outages across 2016 and 2017 by answering three key questions:

    1. What percentage of all outages were Forced Outages? 
    2. How did the average duration of forced outages change between 2016 and 2017? 
    3. Which providers experienced the highest number of forced outage events?

<b>I will use SQL to explore these trends and highlight any emerging reliability concerns.</b>

<h3 style="color:#D4AF37"> Question 6 </h3>

<b style="color:#1708B8">What proportion of total outages were classified as Forced Outages during the 2016–2017 period?
Do we observe any particular increases regarding any Outage Types over this period?</b>

In [695]:
%%sql
SELECT 
    COUNT(*) AS Total_Number_Outages,
    SUM(CASE WHEN Outage_Reason = 'Forced' THEN 1 ELSE 0 END) AS Total_Number_Forced_Outage_Events,
    ROUND(
        (CAST(SUM(CASE WHEN Outage_Reason = 'Forced' THEN 1 ELSE 0 END) AS FLOAT) * 100.0) / COUNT(*),
        2
    ) AS Pct_Outage_Forced
FROM AEMR_Outage_Table
WHERE Year IN (2016, 2017)
  AND Status = 'Approved'
GROUP BY Year
ORDER BY Year;


 * sqlite:///temp_db_file.db
Done.


Total_Number_Outages,Total_Number_Forced_Outage_Events,Pct_Outage_Forced
1931,1264,65.46
2171,1622,74.71


<h3 style="color:#D4AF37"> Question 7 </h3>

<b style="color:#1708B8">Which participants and facilities experienced the most significant operational impact between 2016 and 2017—measured by the number of approved outages, total outage duration, and total energy lost—and who emerges as the highest-risk contributors based on these factors?</b>

In [699]:
%%sql
SELECT
  COUNT(*) AS Total_Number_Outages,
  ROUND(SUM(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))), 2) AS Total_Duration_In_Days,
  ROUND(SUM(COALESCE(Energy_Lost_MW, 0)), 2) AS Total_Energy_Lost,
  Outage_Reason,
  Participant_Code,
  Facility_Code,
  Year
FROM AEMR_Outage_Table
WHERE Status = 'Approved'
  AND Year IN (2016, 2017)
GROUP BY
  Year, Participant_Code, Facility_Code, Outage_Reason
ORDER BY
 Total_Duration_In_Days DESC,
 Total_Number_Outages DESC
LIMIT 5;   


 * sqlite:///temp_db_file.db
Done.


Total_Number_Outages,Total_Duration_In_Days,Total_Energy_Lost,Outage_Reason,Participant_Code,Facility_Code,Year
70,482.58,7499.28,Scheduled (Planned),MELK,MELK_G7,2017
177,404.15,10285.4,Forced,MELK,MELK_G7,2017
85,392.25,9668.79,Scheduled (Planned),MELK,MELK_G7,2016
227,240.69,19326.56,Forced,GW,BW1_GREENWATERS_G2,2017
45,199.4,6450.0,Scheduled (Planned),GW,BW1_GREENWATERS_G2,2016


<h3 style="color:#D4AF37"> Question 8 </h3>

<b style="color:#1708B8">Which participants and facilities experienced the longest and most energy-intensive forced outages during 2016 and 2017, and how do these averages compare across the two years?</b>

In [703]:
%%sql
SELECT
  ROUND(AVG(ABS(JULIANDAY(End_Time) - JULIANDAY(Start_Time))), 2) AS Avg_Duration_In_Days,
  ROUND(AVG(COALESCE(Energy_Lost_MW, 0)), 2) AS Avg_Energy_Lost,
  Outage_Reason,
  Participant_Code,
  Facility_Code,
  year
FROM AEMR_Outage_Table
WHERE Status = 'Approved'
  AND Outage_Reason = 'Forced'
  AND Year IN (2016, 2017)
GROUP BY
  Year, Participant_Code, Facility_Code
ORDER BY
  Avg_Duration_In_Days DESC,
  Avg_Energy_Lost DESC -- sorted by energy lost (highest first)
LIMIT 5;


 * sqlite:///temp_db_file.db
Done.


Avg_Duration_In_Days,Avg_Energy_Lost,Outage_Reason,Participant_Code,Facility_Code,Year
5.9,5.89,Forced,EUCT,GRASMERE_WF1,2016
3.44,27.66,Forced,WGUTD,WEST_KALGOORLIE_GT2,2017
2.28,58.11,Forced,MELK,MELK_G7,2017
2.24,56.32,Forced,ENRG,ENRG_KALGOORLIE_GT3,2016
1.38,61.93,Forced,COLLGAR,COLLGAR_WF1,2017


<h3 style="color:#D4AF37"> Question 9 </h3>

<b style="color:#1708B8"> How much energy is being lost due to forced outages, what share of each facility’s total energy loss do these represent, and which participants are contributing the most to this forced-outage impact? </b>


In [707]:
%%sql
SELECT
  ROUND(AVG(Energy_Lost_MW), 2) AS Avg_Energy_Lost,
  ROUND(SUM(Energy_Lost_MW), 2) AS Total_Energy_Lost,
  ROUND(
    100.0 * SUM(Energy_Lost_MW) /
    (SELECT SUM(Energy_Lost_MW)
     FROM AEMR_Outage_Table
     WHERE Status='Approved'
       AND Year = AEMR_Outage_Table.Year
       AND Facility_Code = AEMR_Outage_Table.Facility_Code),
    2
  ) AS Pct_Energy_Loss,
  Outage_Reason,
  Participant_Code,
  Facility_Code,
  Year
FROM AEMR_Outage_Table
WHERE Status='Approved'
  AND Outage_Reason='Forced'
  AND Year IN (2016, 2017)
GROUP BY Year, Participant_Code, Facility_Code, Outage_Reason
ORDER BY Total_Energy_Lost DESC
LIMIT 6;


 * sqlite:///temp_db_file.db
Done.


Avg_Energy_Lost,Total_Energy_Lost,Pct_Energy_Loss,Outage_Reason,Participant_Code,Facility_Code,Year
44.16,21639.55,8.55,Forced,AURICON,AURICON_PNJ_U1,2017
85.14,19326.56,7.64,Forced,GW,BW1_GREENWATERS_G2,2017
49.69,15751.38,6.23,Forced,GW,BW1_GREENWATERS_G2,2016
87.71,13771.07,5.44,Forced,MELK,MELK_G7,2016
51.42,10696.28,4.23,Forced,AURICON,AURICON_PNJ_U1,2016
58.11,10285.4,4.07,Forced,MELK,MELK_G7,2017


<h3 style="color:#D4AF37"> Question 10 </h3>
<b style="color:#1708B8"> Among the top three energy providers with the greatest energy loss, what was the main outage cause driving their losses, and how much of their total energy loss did it account for?</b>

Having identified the top 3 participants by Total Energy Loss being `GW`, `MELK` and `Auricon`from previous quetion, I calculated totasl energy loss and percent loss


In [645]:
%%sql
WITH t AS (
  SELECT
    Participant_Code,
    Facility_Code,
    Description_Of_Outage,
    SUM(Energy_Lost_MW) AS Total_Energy_Lost,
    RANK () OVER (
      PARTITION BY Participant_Code
      ORDER BY SUM(Energy_Lost_MW) DESC
    ) AS rn
  FROM AEMR_Outage_Table
  WHERE Status='Approved'
    AND Year IN (2016,2017)
    AND Participant_Code IN ('AURICON','GW','MELK')
  GROUP BY Participant_Code, Facility_Code, Description_Of_Outage
)
SELECT
  Participant_Code,
  Facility_Code,
  Description_Of_Outage,
  ROUND(Total_Energy_Lost,2) AS Total_Energy_Lost,
  ROUND(
    100.0 * Total_Energy_Lost /
    (SELECT SUM(Energy_Lost_MW)
     FROM AEMR_Outage_Table
     WHERE Status='Approved'
       AND Year IN (2016,2017)
       AND Participant_Code IN ('AURICON','GW','MELK')
    )
  ,2) AS Pct_Energy_Loss,
  1 AS rank
FROM t
WHERE rn = 1
ORDER BY Participant_Code;


 * sqlite:///temp_db_file.db
Done.


Participant_Code,Facility_Code,Description_Of_Outage,Total_Energy_Lost,Pct_Energy_Loss,rank
AURICON,AURICON_PNJ_U1,Full unit trip,6033.87,4.03,1
GW,BW1_GREENWATERS_G2,Operational Issues caused real time forced outage.,28687.54,19.14,1
MELK,MELK_G7,Safety Issues,1100.0,0.73,1
